In [22]:
import sys
import os
import csv
from gpt_suite import gpt_mp_handler
import pandas as pd
import random
import json
from tqdm.auto import tqdm
from datetime import datetime

In [2]:
norms_df = pd.read_csv('/homes/rpujari/scratch_ml/DARPA/human-in-loop-clustering/flask-gui/norms_data.csv', sep=',')
dialogues_df = pd.read_csv('/homes/rpujari/scratch_ml/DARPA/human-in-loop-clustering/flask-gui/dialogues_data.csv', sep=',')
themes_df = pd.read_csv('/homes/rpujari/scratch_ml/DARPA/human-in-loop-clustering/flask-gui/theme_definitions_v4.csv', sep=',')

In [3]:
norms_df[:3]

,id_,text,distance,good,validated,gpt3_answer_rating,gpt3_feedback,actor_role_id,recepient_role_id,dialogue_id,theme_id
0,0,1. Respect for parents: Filial piety and respe...,0.0501,False,False,[],[],24,24,0,16
1,1,2. Unity within the family: Maintaining harmon...,0.1131,False,False,[],[],24,24,0,14
2,2,3. Social relationships and obligations: Chine...,0.0935,False,False,[],[],24,24,0,14


In [4]:
dialogues_df[:3]

,id_,identifier,culture,dialogue_string,display_text,summary,other_features
0,0,mpdd-1,mandarin-chinese,Mrs. Zuo: What is that foolish girl worth givi...,<b>Conversation Summary:</b><br>The conversati...,The conversation revolves around 左正鵬 (Zho Zpen...,{}
1,1,mpdd-2,mandarin-chinese,"Cousin: Zheng Peng, I heard from the villagers...",<b>Conversation Summary:</b><br>In this conver...,"In this conversation, Zheng Peng's cousin info...",{}
2,2,mpdd-3,mandarin-chinese,"Zuo Zhengpeng: Mom, today is Sunday and I want...",<b>Conversation Summary:</b><br>In this conver...,"In this conversation, Zuo Zhengpeng informs hi...",{}


In [5]:
themes_df[-3:]

,id_,name,description,violation_characteristic,activation_settings,actors,recepients
42,44,RespectForTeachers,"Respecting their knowledge, guidance, and auth...","Disrespecting teachers' authority, disregardin...","school, educational settings, any public settings",any,"teacher, educator"
43,45,SchedulingMeetingTimesInAdvance,Scheduling meeting times in advance shows punc...,"Changing meeting times frequently, arriving la...",any settings,any,any
44,46,ConcernForOthers,Showing concern for others' discomfort or trou...,"Ignoring the troubles of others, not listening...",any settings,any,any


In [6]:
def extract_dialogue_text(display_text):
    display_text = display_text.replace('<br>', '\n').replace('<b>', '').replace('</b>', '')
    dialogue_text_zh = '\n'.join(display_text.split('Dialogue & Translation:')[1].strip().split('\n')[::3])
    return dialogue_text_zh

dialogues_df['dialogue_text_zh'] = dialogues_df['display_text'].apply(extract_dialogue_text)

dialogues_df[:3]

,id_,identifier,culture,dialogue_string,display_text,summary,other_features,dialogue_text_zh
0,0,mpdd-1,mandarin-chinese,Mrs. Zuo: What is that foolish girl worth givi...,<b>Conversation Summary:</b><br>The conversati...,The conversation revolves around 左正鵬 (Zho Zpen...,{},左母(disgust): 那個憨女人有什麼值得送的，正鵬這個人也真是的！\n左父(surpr...
1,1,mpdd-2,mandarin-chinese,"Cousin: Zheng Peng, I heard from the villagers...",<b>Conversation Summary:</b><br>In this conver...,"In this conversation, Zheng Peng's cousin info...",{},堂弟(surprise): 正鵬哥，聽村上人講昨天你娶了個嫂子？\n左正鵬(disgust)...
2,2,mpdd-3,mandarin-chinese,"Zuo Zhengpeng: Mom, today is Sunday and I want...",<b>Conversation Summary:</b><br>In this conver...,"In this conversation, Zuo Zhengpeng informs hi...",{},左正鵬(neutral): 媽，今天是星期天我想去趕下集，可能下午才能回家，有什麼事就別等我...


In [7]:
norm_dialogues_df = pd.merge(norms_df, dialogues_df, left_on='dialogue_id', right_on='id_', how='inner')
print(len(norms_df))
print(len(norm_dialogues_df))
norm_dialogues_df[:3]

63779
63779


,id__x,text,distance,good,validated,gpt3_answer_rating,gpt3_feedback,actor_role_id,recepient_role_id,dialogue_id,theme_id,id__y,identifier,culture,dialogue_string,display_text,summary,other_features,dialogue_text_zh
0,0,1. Respect for parents: Filial piety and respe...,0.0501,False,False,[],[],24,24,0,16,0,mpdd-1,mandarin-chinese,Mrs. Zuo: What is that foolish girl worth givi...,<b>Conversation Summary:</b><br>The conversati...,The conversation revolves around 左正鵬 (Zho Zpen...,{},左母(disgust): 那個憨女人有什麼值得送的，正鵬這個人也真是的！\n左父(surpr...
1,1,2. Unity within the family: Maintaining harmon...,0.1131,False,False,[],[],24,24,0,14,0,mpdd-1,mandarin-chinese,Mrs. Zuo: What is that foolish girl worth givi...,<b>Conversation Summary:</b><br>The conversati...,The conversation revolves around 左正鵬 (Zho Zpen...,{},左母(disgust): 那個憨女人有什麼值得送的，正鵬這個人也真是的！\n左父(surpr...
2,2,3. Social relationships and obligations: Chine...,0.0935,False,False,[],[],24,24,0,14,0,mpdd-1,mandarin-chinese,Mrs. Zuo: What is that foolish girl worth givi...,<b>Conversation Summary:</b><br>The conversati...,The conversation revolves around 左正鵬 (Zho Zpen...,{},左母(disgust): 那個憨女人有什麼值得送的，正鵬這個人也真是的！\n左父(surpr...


In [8]:
def make_norm_prompts(row):
    prompt = f"Norm ID: {row['id__x']}\nConversation from Chinese culture:\n{row['dialogue_text_zh'].strip()}\n\nSocial Norm:\n{row['text'].strip()}"
    return prompt

norm_dialogues_df['norm_prompt'] = norm_dialogues_df.apply(make_norm_prompts, axis=1)

print(norm_dialogues_df[:1].norm_prompt.iloc[0])

Norm ID: 0
Conversation from Chinese culture:
左母(disgust): 那個憨女人有什麼值得送的，正鵬這個人也真是的！
左父(surprise): 哎喲，老婆子，你怎麼盡講那些不利於團結的話呢！他去送送他的同學也在情理之中嘛！
左正鵬(neutral): 爸、媽，我回來啦！
左母(disgust): 我怕你喝了迷魂湯，魂被你那個憨同學勾引去了！
左父(surprise): 老婆子，看你講話像個母親說的嗎！你怎麼老是反對他們倆的事呢？
左正鵬(neutral): 爸、媽，你們都不要講了。這件事我自己有主見的，我知道自己該怎麼去做，不應該怎麼不去做。決不會胡來的，請您們放心好了！

Social Norm:
1. Respect for parents: Filial piety and respect for parents are highly valued in Chinese culture. Children are expected to listen to and obey their parents' opinions and decisions.


In [9]:
openai_key = '<put-your-key-here>'
# openai_key = '<put-your-key-here>'

In [10]:
debug_dir = '/homes/rpujari/scratch_ml/DARPA/automated_verification/norm_verification_logs/'
os.makedirs(debug_dir, exist_ok=True)

In [11]:
DEFAULT_SYSTEM_PROMPT = 'You are a helpful assistant. Your task is to judge the relevance of a Chinese cultural social norm to the sitatution in a given conversation.'+\
' Consider factors such as age of the people involved, relationships between them, settings of the conversations such as work, family or friends, topic of conversation and so on.'+\
' Respond with "relevant"/"irrelevant" label and provide a justifcation for your decision. Strictly format your answer as: Justification: <justification>\nRelevance: <decision>'


def norm_verifier(norm_prompts) -> list:
    config = {"temperature": 0, "max_tokens": 500}
    model = 'gpt-4o-mini'
    handler = gpt_mp_handler.GPTMPHandler(api_key=openai_key, gen_conf=config, num_worker=10)
    results = list()
    batch = []
    for norm_id in norm_prompts:
        ins = {
            'init_context': '',
            'questions': [norm_prompts[norm_id]],
            'task_desc': DEFAULT_SYSTEM_PROMPT,
            'debug_log': debug_dir,
            'model_name': model
        }
        batch.append(ins)
        results.append([norm_id])
    handler.add_batch(batch)
    outs = handler.process()

    for idx, out in enumerate(outs):
        if len(out) == 0:
            print("ERROR:Missing...")
            results[idx].append(-1)
            continue
        results[idx].append(list(out.items())[-1][1])
    return results

In [12]:
norm_prompts = norm_dialogues_df[['id__x', 'norm_prompt']].set_index('id__x')['norm_prompt'].to_dict()
for k, v in norm_prompts.items():
    toss = random.random()
    if toss >= 0.9:
        print(k)
        print(v)
        break

38
Norm ID: 38
Conversation from Chinese culture:
徐麗華(angry): 孫校長，昨天晚上左正鵬把家裏的錢給生產隊買耕牛去了，弄得家裏沒有一分錢；我跟他母親頂嘴，他又說我沒有教養；我去合作社扯布也去不成了。你說，我的氣往哪兒消呀！他還是不是我的男人？
孫校長(neutral): 小徐呀！這是件很小的事情。你別生氣。我叫左老師放學後給你溝通溝通下就行啦！哪有夫妻倆不吵架的？快回去啊！站在這裏不好看。
徐麗華(angry): 孫校長，就拜託了，你要給他嚴厲地批評批評啊！使他心服口服。
孫校長(neutral): 你們都要多作自我批評，少說別人缺點，要放下電筒拿起鏡子。你快點回去吧。

Social Norm:
3. Collectivism and filial piety: The concept of family is valued, and Xu Lihua's frustration with her husband's actions may be seen as a violation of the harmony and cooperation within the family unit.


In [23]:
json.dump([DEFAULT_SYSTEM_PROMPT, norm_prompts], open('/homes/rpujari/scratch_ml/DARPA/automated_verification/norm_prompts.json', 'w'))

In [13]:
norm_prompt_keys = sorted(list(norm_prompts.keys()))

In [14]:
batch_id = 0
batches = []
b = 0
bsz = 1000
e = bsz
while b < len(norm_prompt_keys):
    batch_keys = norm_prompt_keys[b:e]
    batch = {}
    for bkey in batch_keys:
        batch[bkey] = norm_prompts[bkey]
    batches.append(batch)
    b += bsz
    e += bsz

In [15]:
output_dir = '/homes/rpujari/scratch_ml/DARPA/automated_verification/norm_verification_outputs/'
os.makedirs(output_dir, exist_ok=True)

t1 = datetime.now()
for i, batch in tqdm(enumerate(batches)):
    batch_out_path = os.path.join(output_dir, f"batch_{i}.json")
    if not os.path.exists(batch_out_path):
        batch_res = norm_verifier(batch)
        json.dump(batch_res, open(batch_out_path, 'w'))
    t2 = datetime.now()
    print(f'batch {i} done.', t2-t1)

0it [00:00, ?it/s]

batch 0 done. 0:00:00.025600
batch 1 done. 0:00:00.026019
batch 2 done. 0:00:00.026224
batch 3 done. 0:00:00.026464
batch 4 done. 0:00:00.026712
batch 5 done. 0:00:00.026968
batch 6 done. 0:00:00.027220
batch 7 done. 0:00:00.027474
batch 8 done. 0:00:00.027750
batch 9 done. 0:00:00.027984
batch 10 done. 0:00:00.028232
batch 11 done. 0:00:00.028485
batch 12 done. 0:00:00.028739
batch 13 done. 0:00:00.028993
batch 14 done. 0:00:00.029248
batch 15 done. 0:00:00.029509
batch 16 done. 0:00:00.029758
batch 17 done. 0:00:00.030005
batch 18 done. 0:00:00.030258
batch 19 done. 0:00:00.030519
batch 20 done. 0:00:00.030765
batch 21 done. 0:00:00.031017
batch 22 done. 0:00:00.031278
batch 23 done. 0:00:00.031544
batch 24 done. 0:00:00.031811
batch 25 done. 0:00:00.032076
batch 26 done. 0:00:00.032289
batch 27 done. 0:00:00.032553
batch 28 done. 0:00:00.032796


Verifying Batch:   0%|          | 0/1000 [00:00<?, ?it/s]

Processing Batch:   0%|          | 0/1000 [00:00<?, ?it/s]

batch 29 done. 0:06:29.708139


...

Processing Batch:   0%|          | 0/1000 [00:00<?, ?it/s]

batch 62 done. 3:03:57.026697


Verifying Batch:   0%|          | 0/779 [00:00<?, ?it/s]

Processing Batch:   0%|          | 0/779 [00:00<?, ?it/s]

batch 63 done. 3:08:41.445400


In [20]:
w = 0
t = 0
rem_batch = {}
ann_ids = set()
for i, batch in tqdm(enumerate(batches)):
    if os.path.exists(os.path.join(output_dir, f'batch_{i}.json')):
        batch_res = json.load(open(os.path.join(output_dir, f'batch_{i}.json')))
        for res in batch_res:
            if len(res) != 2 or (not str(res[1]).strip().lower().startswith('justification')):
                w += 1
                rem_batch[res[0]] = norm_prompts[res[0]]
            else:
                ann_ids.add(res[0])
        t += len(batch_res)
print(w, t, len(rem_batch), len(ann_ids))

0it [00:00, ?it/s]

0 63779 0 63779
